In [1]:
import normalize as n

In [2]:
con = n.connect("songs.duckdb", memory_limit="4GB")

In [3]:
n.load_csv(con, "df", "./data/1mil2-isrc.csv", {"song": "song", "artists": "artists", "year": "year", "isrc": "isrc"})
n.load_csv(con, "df_hot", "./data/hot-100-2000.csv", {"song": "song", "artists": "artists", "isrc": "isrc"})
n.load_csv(con, "lyr", "./data/song_lyrics.csv", {"title": "song", "artist": "artist", "features": "features", "year": "year"})

df: 1,204,025 rows loaded
df_hot: 114,100 rows loaded
lyr: 5,123,643 rows loaded


In [4]:
n.normalize_df(con)
n.normalize_hot(con)
n.normalize_lyr(con)

  normalizing 165,365 distinct strings of df.artists
  df_v: 2,342,087 (row, variant) pairs
  normalizing 4,200 distinct strings of df_hot.artists
  df_hot_v: 417,353 (row, variant) pairs
  normalizing 640,509 distinct strings of lyr.artist
  normalizing 450,263 distinct strings of lyr.features
  lyr_v: 12,432,284 (row, variant) pairs


In [5]:
n.match_count(con, "df_hot", "df", "matches_hot", use_year=False)

df_hot rows: 114,100
  title + artist only      : 28,122 (24.6%)
  (final)                  : 28,122 (24.6%)
  unique songs matched     : 1,898 / 9,197 (20.6%)
  right rows per matched left row: median 1, max 13


{'rows': 114100, 'title_artist': 28122, 'final': 28122, 'songs': 1898}

In [6]:
n.match_count(con, "df", "lyr", "matches_lyr", use_year=True, year_tol=1)

df rows: 1,204,025
  title + artist only      : 219,350 (18.2%)
  + same year              : 148,374 (12.3%)
  + year within +/-1       : 163,408 (13.6%)
  unique songs matched     : 153,576 / 1,139,110 (13.5%)
  right rows per matched left row: median 1, max 4


{'rows': 1204025, 'title_artist': 219350, 'final': 163408, 'songs': 153576}

In [7]:
n.match_report(con, "df", "matches_lyr", year_tol=2)

df rows: 1,204,025
  title + artist only      : 219,350 (18.2%)
  + same year              : 148,374 (12.3%)
  + year within +/-2       : 168,375 (14.0%)
  unique songs matched     : 157,359 / 1,139,110 (13.8%)
  right rows per matched left row: median 1, max 4


{'rows': 1204025, 'title_artist': 219350, 'final': 168375, 'songs': 157359}

In [8]:
n.isrc_check(con, "df_hot", "df", "matches_hot")

Title+artist matches: 43,813 pairs, 28,122 df_hot rows
Precision (does ISRC agree?)
  pairs where both sides have an ISRC : 42,749 (97.6%)
  ...of those, same ISRC              : 27,967 (65.4%)
  ...of those, different ISRC         : 14,782 (34.6%)  -> matches_hot_mismatch
  df_hot rows checkable               : 27,388 (97.4%)
  ...with at least one same-ISRC pair : 20,232 (73.9%)
Recall (does title+artist find the ISRC matches?)
  pairs sharing an ISRC               : 40,082
  ...also found by title+artist       : 27,967 (69.8%)
  df_hot rows with an ISRC match      : 28,760
  ...also found by title+artist       : 21,112 (73.4%)  (missed ones: matches_hot_isrc pairs not in matches_hot)


{'pairs': 43813,
 'checkable': 42749,
 'same': 27967,
 'isrc_pairs': 40082,
 'isrc_pairs_found': 27967}

In [10]:
con.close()

## Merge
Only isrc+title+artists matches for hot, title+artist year_tol=0 for lyr

# New normalization for base_full and lyrics

In [2]:
import normalize1 as n1
con = n1.run()

== load
base: 161,916 rows loaded
  lyr: reusing existing table (5,123,643 rows)
== normalizing base
  base_keys1: 162,726 keys for 158,032 / 161,916 rows (3,884 rows have no usable key)
== normalizing lyr
  lyr_keys1: 5,259,298 keys for 5,036,185 / 5,123,643 rows (87,458 rows have no usable key)
== matching
  matches1: 92,151 (base row, lyr row) pairs
base rows: 161,916 (7,788 hits, 154,128 non-hits)
                                       all              hits          non-hits
title + artist only         85,818 (53.0%)     7,092 (91.1%)    78,726 (51.1%)
+ same year                 57,915 (35.8%)     5,413 (69.5%)    52,502 (34.1%)
+ year within +/-1          65,984 (40.8%)     6,769 (86.9%)    59,215 (38.4%)
+ year within +/-2          68,324 (42.2%)     6,929 (89.0%)    61,395 (39.8%)
lyr rows per matched base row (within +/-2): median 1, max 8


In [3]:
n1.unmatched(con, limit=30, hits_only=True).show()
n1.report(con, max_year_diff=3)

┌─────────┬─────────┬───────┬─────────┐
│  song   │ artist  │ year  │   hit   │
│ varchar │ varchar │ int32 │ varchar │
└─────────┴─────────┴───────┴─────────┘
                0 rows               

base rows: 161,916 (7,788 hits, 154,128 non-hits)
                                       all              hits          non-hits
title + artist only         85,818 (53.0%)     7,092 (91.1%)    78,726 (51.1%)
+ same year                 57,915 (35.8%)     5,413 (69.5%)    52,502 (34.1%)
+ year within +/-1          65,984 (40.8%)     6,769 (86.9%)    59,215 (38.4%)
+ year within +/-2          68,324 (42.2%)     6,929 (89.0%)    61,395 (39.8%)
+ year within +/-3          69,754 (43.1%)     6,952 (89.3%)    62,802 (40.7%)
lyr rows per matched base row (within +/-3): median 1, max 8


# Merge on base_full

In [1]:
import normalize as n
import normalize1 as n1

OUT_CSV = "./data/base_full_lyrics.csv"
LYR_LYRICS_HEADER = "lyrics"
LYR_LANG_HEADER = "language_ft"
# ---------------------------------------------------------------------------

q = lambda s: '"' + s.replace('"', '""') + '"'
lit = lambda s: str(s).replace("'", "''")
READ_OPTS = "header=true, all_varchar=true, ignore_errors=true"
base_id_hdr = next(k for k, v in n1.BASE_COLS.items() if v == "src_id")   
lyr_id_hdr = next(k for k, v in n1.LYR_COLS.items() if v == "src_id")    

con = n.connect(n1.DB_PATH, memory_limit=n1.MEMORY_LIMIT)

for t in ("base", "lyr", "matches1"):
    assert n._table_exists(con, t), f"table {t} missing: run normalize1.run() first"
dup = con.execute("SELECT count(*) - count(DISTINCT src_id) FROM base").fetchone()[0]
assert dup == 0, f"{dup:,} duplicated ids in base_full.csv, the merge would duplicate rows"

con.execute("""
    CREATE OR REPLACE TABLE base_lyr_pick AS
    SELECT base_id, lyr_src_id, n_matches FROM (
        SELECT m.base_id, l.src_id AS lyr_src_id,
               count(*) OVER (PARTITION BY m.base_id) AS n_matches,
               row_number() OVER (PARTITION BY m.base_id
                                  ORDER BY m.year_diff NULLS LAST, l.src_id) AS rk
        FROM matches1 m
        JOIN lyr l ON l.id = m.lyr_id
    ) WHERE rk = 1
""")
con.execute("""
    CREATE OR REPLACE TABLE base_lyr_extra AS
    SELECT b.src_id, p.lyr_src_id AS lyr_id, p.n_matches AS lyr_n_matches
    FROM base b JOIN base_lyr_pick p ON p.base_id = b.id
""")

lid = q(lyr_id_hdr)
con.execute(f"""
    CREATE OR REPLACE TABLE base_lyr_payload AS
    SELECT r.{lid} AS lyr_id, r.{q(LYR_LYRICS_HEADER)} AS lyrics, r.{q(LYR_LANG_HEADER)} AS language_ft
    FROM read_csv('{lit(n1.LYR_CSV)}', {READ_OPTS}) r
    WHERE r.{lid} IN (SELECT lyr_id FROM base_lyr_extra)
""")

con.execute("SET preserve_insertion_order=true")
con.execute(f"""
    COPY (
        SELECT d.* EXCLUDE (_rn), e.lyr_id, e.lyr_n_matches, p.lyrics, p.language_ft
        FROM (SELECT row_number() OVER () AS _rn, * FROM read_csv('{lit(n1.BASE_CSV)}', {READ_OPTS})) d
        LEFT JOIN base_lyr_extra e ON e.src_id = d.{q(base_id_hdr)}
        LEFT JOIN base_lyr_payload p ON p.lyr_id = e.lyr_id
        ORDER BY d._rn
    ) TO '{lit(OUT_CSV)}' (HEADER, DELIMITER ',')
""")
con.execute("SET preserve_insertion_order=false")

s = con.execute("""
    SELECT count(*), count(*) FILTER (WHERE TRY_CAST(b.hit AS INTEGER) = 1),
           count(e.lyr_id), count(e.lyr_id) FILTER (WHERE TRY_CAST(b.hit AS INTEGER) = 1)
    FROM base b LEFT JOIN base_lyr_extra e ON e.src_id = b.src_id""").fetchone()
print(f"{OUT_CSV}: {s[0]:,} rows | with lyrics {s[2]:,} ({s[2] / s[0]:.1%}) | "
      f"hits {s[3]:,}/{s[1]:,} ({s[3] / max(s[1], 1):.1%})")

con.close()

./data/base_full_lyrics.csv: 161,916 rows | with lyrics 85,818 (53.0%) | hits 7,092/7,788 (91.1%)
